In [1]:
# ============================================================
# Project: EuroTrans Analytics
# Notebook: 02_Silver_Transformation
# Layer: Silver
#
# Description:
# Clean and standardize Bronze tables before
# publishing them to the Silver layer.
# ============================================================

# ------------------------------------------------------------
# Import Libraries
# ------------------------------------------------------------

from pyspark.sql import SparkSession
from pyspark.sql.functions import trim, col

# ------------------------------------------------------------
# Create Spark Session
# ------------------------------------------------------------

spark = SparkSession.builder.getOrCreate()

# ------------------------------------------------------------
# Bronze Tables
# ------------------------------------------------------------

bronze_tables = [
    "dim_customer",
    "dim_product",
    "dim_date",
    "dim_route",
    "dim_carrier",
    "dim_warehouse",
    "fact_shipment",
    "fact_fuel_price",
    "fact_road_event"
]

print("=" * 60)
print("Starting Silver Transformation...")
print("=" * 60)

# ------------------------------------------------------------
# Process Bronze Tables
# ------------------------------------------------------------

for table in bronze_tables:

    print(f"\nProcessing: {table}")

    # Load Bronze table
    df = spark.table(table)

    rows_before = df.count()

    # Remove duplicate records
    df = df.dropDuplicates()

    # Trim all string columns
    for field in df.schema.fields:
        if field.dataType.simpleString() == "string":
            df = df.withColumn(
                field.name,
                trim(col(field.name))
            )

    rows_after = df.count()

    silver_table = (
        table
        .replace("dim_", "silver_")
        .replace("fact_", "silver_")
    )

    (
        df.write
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .format("delta")
            .saveAsTable(silver_table)
    )

    print(f"✓ Silver table: {silver_table}")
    print(f"Rows: {rows_after}")
    print(f"Columns: {len(df.columns)}")
    print(f"Duplicates removed: {rows_before - rows_after}")

print("\n" + "=" * 60)
print("Silver Layer successfully created.")
print("=" * 60)

StatementMeta(, 3c6ce3c0-a792-437d-ab86-e1208e5328b3, 3, Finished, Available, Finished, True)

Starting Silver Transformation...

Processing: dim_customer
✓ Silver table: silver_customer
Rows: 800
Columns: 6
Duplicates removed: 0

Processing: dim_product
✓ Silver table: silver_product
Rows: 7
Columns: 7
Duplicates removed: 0

Processing: dim_date
✓ Silver table: silver_date
Rows: 731
Columns: 15
Duplicates removed: 0

Processing: dim_route
✓ Silver table: silver_route
Rows: 25
Columns: 6
Duplicates removed: 0

Processing: dim_carrier
✓ Silver table: silver_carrier
Rows: 7
Columns: 5
Duplicates removed: 0

Processing: dim_warehouse
✓ Silver table: silver_warehouse
Rows: 10
Columns: 7
Duplicates removed: 0

Processing: fact_shipment
✓ Silver table: silver_shipment
Rows: 50000
Columns: 17
Duplicates removed: 0

Processing: fact_fuel_price
✓ Silver table: silver_fuel_price
Rows: 7310
Columns: 3
Duplicates removed: 0

Processing: fact_road_event
✓ Silver table: silver_road_event
Rows: 500
Columns: 5
Duplicates removed: 0

Silver Layer successfully created.
